In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg
!pip install torch torchaudio librosa soundfile tqdm requests

import os
import zipfile
import requests
import shutil
from pathlib import Path
from google.colab import files

import librosa
import soundfile as sf
import torch
import numpy as np
from tqdm import tqdm

def download_yandex_disk(url, output_path):
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

    r = requests.get(api_url, params={"public_key": url})
    download_url = r.json()["href"]

    with requests.get(download_url, stream=True) as r:
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)


def unzip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)


SR = 16000
all_lengths = []

MIN_LEN = 2.0
MAX_LEN = 20.0
MAX_LEN_HARD = 25.0

MERGE_GAP = 0.8
SPLIT_MIN_SILENCE = 0.5

WINDOW = 12.0
OVERLAP = 0.8

FRAME = 0.02

model, utils = torch.hub.load(
    'snakers4/silero-vad',
    'silero_vad',
    trust_repo=True
)

(get_speech_timestamps, _, _, _, _) = utils

def load_audio(path):
    audio, _ = librosa.load(path, sr=SR, mono=True)
    return audio

def vad_segments(audio):
    ts = get_speech_timestamps(
        torch.tensor(audio),
        model,
        sampling_rate=SR
    )
    return [(t['start']/SR, t['end']/SR) for t in ts]

def duration(seg):
    return seg[1] - seg[0]

def gap(a, b):
    return b[0] - a[1]

def normalize(segs):
    segs = sorted(segs)
    out = []
    for s in segs:
        if not out:
            out.append(s)
            continue
        prev = out[-1]
        if s[0] <= prev[1]:
            out[-1] = (prev[0], max(prev[1], s[1]))
        else:
            out.append(s)
    return out

def merge_short_once(segs):
    segs = sorted(segs)
    result = []

    i = 0
    while i < len(segs):
        cur = segs[i]

        if duration(cur) >= MIN_LEN:
            result.append(cur)
            i += 1
            continue

        prev_seg = result[-1] if result else None
        next_seg = segs[i+1] if i+1 < len(segs) else None

        merged = False

        if next_seg and gap(cur, next_seg) <= MERGE_GAP:
            segs[i+1] = (cur[0], next_seg[1])
            merged = True

        elif prev_seg and gap(prev_seg, cur) <= MERGE_GAP:
            result[-1] = (prev_seg[0], cur[1])
            merged = True

        i += 1

    return normalize(result)

def merge_until_stable(segs, max_iter=10):
    for _ in range(max_iter):
        new = merge_short_once(segs)
        if new == segs:
            break
        segs = new
    return segs


def find_silences(audio, start, end):
    s = int(start * SR)
    e = int(end * SR)

    chunk = audio[s:e]

    frame_len = int(FRAME * SR)
    hop = frame_len

    energies = []
    for i in range(0, len(chunk) - frame_len, hop):
        frame = chunk[i:i+frame_len]
        energies.append(np.mean(frame**2))

    if len(energies) == 0:
        return []

    energies = np.array(energies) + 1e-10
    log_e = np.log(energies)

    thresh = np.percentile(log_e, 20)

    silences = []
    start_idx = None

    for i, val in enumerate(log_e):
        if val < thresh and start_idx is None:
            start_idx = i
        elif val >= thresh and start_idx is not None:
            dur = (i - start_idx) * FRAME
            if dur >= SPLIT_MIN_SILENCE:
                silences.append((
                    start + start_idx * FRAME,
                    start + i * FRAME
                ))
            start_idx = None

    return silences


def split_long(audio, seg):
    start, end = seg
    dur = duration(seg)

    if dur <= MAX_LEN:
        return [seg]

    silences = find_silences(audio, start, end)

    # 1. split по паузам
    if silences:
        points = [(s + e) / 2 for s, e in silences]

        parts = []
        cur = start

        for p in points:
            if p - cur >= MIN_LEN:
                parts.append((cur, p))
                cur = p

        if end - cur >= MIN_LEN:
            parts.append((cur, end))

        if len(parts) > 1:
            return parts

    # 2. fallback
    parts = []
    cur = start

    while cur < end:
        seg_end = min(cur + WINDOW, end)
        parts.append((cur, seg_end))
        cur += WINDOW - OVERLAP

    return parts


def refine_segments(segs):
    # итеративный merge коротких
    segs = merge_until_stable(segs)

    # удаляем совсем мусор (очень короткие после всех попыток)
    segs = [s for s in segs if duration(s) > 0.3]

    return segs


def process_audio(audio):

    segs = vad_segments(audio)

    segs = refine_segments(segs)

    split_segs = []
    for seg in segs:
        if duration(seg) > MAX_LEN:
            split_segs.extend(split_long(audio, seg))
        else:
            split_segs.append(seg)

    split_segs = refine_segments(split_segs)

    # финальная нормализация
    split_segs = normalize(split_segs)

    # финальный guard
    final = []
    for s in split_segs:
        d = duration(s)
        if MIN_LEN <= d <= MAX_LEN_HARD:
            final.append(s)

    return final


def process_dataset(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    audio_ext = (".wav", ".mp3", ".flac", ".ogg")

    files = []
    for root, _, fs in os.walk(input_dir):
        for f in fs:
            if f.lower().endswith(audio_ext):
                files.append(os.path.join(root, f))

    print(f"Found {len(files)} files")

    for path in tqdm(files):
        try:
            audio = load_audio(path)
            segs = process_audio(audio)

            base = Path(path).stem

            for i, (s, e) in enumerate(segs):
                chunk = audio[int(s*SR):int(e*SR)]
                out = os.path.join(output_dir, f"{base}_{i}.wav")
                sf.write(out, chunk, SR)
                all_lengths.append(e - s)

        except Exception as e:
            print(f"Error: {path} → {e}")


input_dir = "data"
output_dir = "processed"

yandex_url = "YANDEX DISK LINK"
zip_path = "non-urmi_unlabeled.zip"

download_yandex_disk(yandex_url, zip_path)
unzip(zip_path, "data")

process_dataset(input_dir, output_dir)

if all_lengths:
    print("\nGLOBAL STATISTICS")
    print(f"Min segment length: {min(all_lengths):.2f} sec")
    print(f"Max segment length: {max(all_lengths):.2f} sec")
else:
    print("No segments found")

# ZIP

zip_name = "2_non-urmi_unlabeled"
output_dir = "processed"

if os.path.exists(zip_name):
    shutil.rmtree(zip_name)

if os.path.exists(zip_name + ".zip"):
    os.remove(zip_name + ".zip")

shutil.move(output_dir, zip_name)

shutil.make_archive(zip_name, 'zip', zip_name)

files.download(zip_name + ".zip")